## AfroXLMR - Results Summary

- **Model:** Davlan/afro-xlmr-base (270M parameters)
- **Max sequence length:** 64
- **Optimizer:** AdamW (lr=0.00002)
- **Scheduler:** Linear warmup (10% warmup steps)
- **Class weights:** CrossEntropyLoss with balanced class weights
- **Gradient clipping:** max_norm=1.0
- **Preprocessing:** remove URLs only

**Training Progress:**
- Epoch 1:  Val Macro F1 0.73  |  Val Accuracy 0.73
- Epoch 2:  Val Macro F1 0.76  |  Val Accuracy 0.76
- Epoch 3:  Val Macro F1 0.77  |  Val Accuracy 0.77  ← best, saved

**Overall Test Results:**
- AfroXLMR:  Macro F1 0.74  |  Accuracy 0.74

**Per-Language Results:**
- Hausa  achieved a Macro F1 of 0.77 — beat LSTM (0.75 → 0.77)  - improved
- Igbo   achieved a Macro F1 of 0.78 — beat LSTM (0.75 → 0.78)  - improved
- Pidgin achieved a Macro F1 of 0.49 — still the worst (Pidgin neutral F1: 0.07)
- Yoruba achieved a Macro F1 of 0.70 — beat LSTM (0.69 → 0.70)  - improved

**Key Findings:**
- AfroXLMR beat every previous model (0.74 vs 0.71 LSTM)
- Beat LSTM on Hausa, Igbo, and Yoruba — pretrained African language knowledge works
- Yoruba diacritics handled natively by XLM-R tokenizer — advantage over Keras tokenizer
- Pidgin neutral still collapsed (F1: 0.07) — 72 training samples is a data problem, not a model problem
- Validation F1 (0.77) vs Test F1 (0.74) — 3-point gap is normal, no overfitting concern

**vs LSTM Baseline:**
- Overall Macro F1:  0.74 vs 0.71  ← AfroXLMR leads
- Hausa  F1:  0.77 vs 0.75         ← AfroXLMR leads
- Igbo   F1:  0.78 vs 0.75         ← AfroXLMR leads
- Pidgin F1:  0.49 vs 0.43         ← AfroXLMR leads
- Yoruba F1:  0.70 vs 0.69         ← AfroXLMR leads

**AfroXLMR beat LSTM on all 4 languages. Best model for deployment.**

In [1]:
# import the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup

from utils import dataset_preprocessing_afro_xlmr
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

In [2]:
# load the datasets
df_train = pd.read_csv('../01-data/02-processed/train_clean.csv')
df_test = pd.read_csv('../01-data/02-processed/test_clean.csv')
df_val = pd.read_csv('../01-data/02-processed/val_clean.csv')

In [3]:
# drop tweet length from train dataset
df_train.drop(columns=['tweet_length'], inplace=True)

df_list = [df_train, df_test, df_val]
for df in df_list:
    df['cleaned_tweet'] = df['tweet'].apply(dataset_preprocessing_afro_xlmr)
    df.drop(columns=['tweet'], inplace=True)
    df.rename(columns={'cleaned_tweet': 'tweet'}, inplace=True)

In [4]:
# split dataset for training, validation and testing
X_train = df_train['tweet']
y_train = df_train['label']
X_test = df_test['tweet']
y_test = df_test['label']
X_val = df_val['tweet']
y_val = df_val['label']

In [5]:
label_mapping = {'negative': 0, 'neutral': 1, 'positive': 2}
y_train = y_train.map(label_mapping).values
y_val   = y_val.map(label_mapping).values
y_test  = y_test.map(label_mapping).values

In [6]:
# load tokenizer

tokenizer = AutoTokenizer.from_pretrained("Davlan/afro-xlmr-base")

In [7]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [8]:
# Create Dataset & DataLoader
dataset = TextDataset(list(X_train), y_train, tokenizer, max_len=64)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True
)

val_dataset = TextDataset(list(X_val), y_val, tokenizer, max_len=64)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False
)

In [9]:
# load the model

model = AutoModelForSequenceClassification.from_pretrained(
    "Davlan/afro-xlmr-base",
    num_labels=3
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=768, out_features=3, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Li

In [11]:
# optimizer

optimizer = AdamW(model.parameters(), lr=0.00002)

In [12]:
classes = np.array([0, 1, 2])
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights_tensor = torch.FloatTensor(weights).to(device)

# weighted loss would be used instead of model's internal loss
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

In [ ]:
# Training loop

epochs = 3

best_f1 = 0.0

total_steps = len(loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_batch = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_batch
        )

        loss = loss_fn(outputs.logits, labels_batch)
        total_loss += loss.item()


        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits

            # Convert to predicted class
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Metrics
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')

    # Compute average loss
    avg_loss = total_loss / len(loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {avg_loss:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print("\nClassification Report:\n")
    print(classification_report(all_labels, all_preds, target_names=['negative', 'neutral', 'positive']))

    # save the best model
    if f1 > best_f1:

        ]
        ]]



        
        best_f1 = f1
        torch.save(model.state_dict(), f"../03-models/best_afroxlmr_model_f1_{best_f1:.4f}.pt")
        print(f"Best model saved with F1: {best_f1:.4f}")



Epoch 1
Loss: 0.7970
Accuracy: 0.7333
F1 Score: 0.7316

Classification Report:

              precision    recall  f1-score   support

    negative       0.73      0.67      0.70      2434
     neutral       0.70      0.75      0.72      2489
    positive       0.77      0.78      0.77      2676

    accuracy                           0.73      7599
   macro avg       0.73      0.73      0.73      7599
weighted avg       0.73      0.73      0.73      7599

Best model saved with F1: 0.7316

Epoch 2
Loss: 0.5821
Accuracy: 0.7577
F1 Score: 0.7560

Classification Report:

              precision    recall  f1-score   support

    negative       0.73      0.74      0.74      2434
     neutral       0.76      0.71      0.73      2489
    positive       0.78      0.82      0.80      2676

    accuracy                           0.76      7599
   macro avg       0.76      0.76      0.76      7599
weighted avg       0.76      0.76      0.76      7599

Best model saved with F1: 0.7560

Epoch 3
L

In [14]:
test_dataset = TextDataset(list(X_test), y_test, tokenizer, max_len=64)
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Evaluate on test set
model.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

f1 = f1_score(all_labels, all_preds, average='macro')
print(f'Test Macro F1: {f1}')
print(classification_report(all_labels, all_preds,
      target_names=['negative', 'neutral', 'positive']))

Test Macro F1: 0.7396977255611187
              precision    recall  f1-score   support

    negative       0.69      0.78      0.73      6009
     neutral       0.75      0.67      0.71      5457
    positive       0.79      0.77      0.78      6188

    accuracy                           0.74     17654
   macro avg       0.74      0.74      0.74     17654
weighted avg       0.74      0.74      0.74     17654



In [15]:
reverse_mapping = {0: 'negative', 1: 'neutral', 2: 'positive'}
df_test['predicted'] = [reverse_mapping[p] for p in all_preds]

for lang in ['hausa', 'igbo', 'pidgin', 'yoruba']:
    subset = df_test[df_test['language'] == lang]
    print(f"\n{'='*50}")
    print(f"Language: {lang.upper()}")
    print(f"{'='*50}")
    print(classification_report(subset['label'], subset['predicted'],
                                target_names=['negative', 'neutral', 'positive']))


Language: HAUSA
              precision    recall  f1-score   support

    negative       0.73      0.79      0.76      1759
     neutral       0.76      0.67      0.71      1789
    positive       0.83      0.86      0.84      1755

    accuracy                           0.77      5303
   macro avg       0.77      0.77      0.77      5303
weighted avg       0.77      0.77      0.77      5303


Language: IGBO
              precision    recall  f1-score   support

    negative       0.75      0.69      0.71       943
     neutral       0.76      0.80      0.78      1621
    positive       0.83      0.83      0.83      1118

    accuracy                           0.78      3682
   macro avg       0.78      0.77      0.78      3682
weighted avg       0.78      0.78      0.78      3682


Language: PIDGIN
              precision    recall  f1-score   support

    negative       0.70      0.90      0.78      2326
     neutral       0.42      0.04      0.07       431
    positive       0.70 